# LLM Alpha Loop — Dataset-Scoped Field Selection

1. **Cell 2**: type the dataset name → system crawls ALL its fields for the region/universe.
2. **Cell 3**: pick a subset of those fields (indices, IDs, or all) for the agent.
3. **Cell 4**: LLM proposes expressions using ONLY your subset (VECTOR fields auto-flagged for `vec_avg` wrapping).
4. **Cell 5**: real simulations → results appended to history → loop repeats with the full history in the prompt.

Re-run cell 5 (then cell 6) to keep iterating on the same history.

In [17]:
import os, json, getpass, functools, time
from urllib.parse import urljoin
import pandas as pd
import ace_lib as ace
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from pathlib import Path


class SessionExpired(RuntimeError):
    """Raised when BRAIN auth expires mid-run - caught by the RUN loop."""

def check_session_and_relogin_safe(sess):
    """Non-blocking replacement for ace.check_session_and_relogin.
    Never prompts for biometrics mid-run: fails LOUD and lets the RUN loop
    save state and tell you to re-auth on your terms."""
    with sess.get_relogin_lock():
        rem = ace.check_session_timeout(sess)
        if rem < 2000:
            raise SessionExpired(
                f'BRAIN session expires in ~{rem:.0f}s -> NOT auto-re-logging-in.\n'
                '1) Interrupt this cell.  2) Re-run the Setup cell and complete '
                'biometrics at the FRESH url.\n'
                '3) Re-run the RUN cell - it resumes from history automatically.')
    return sess

ace.check_session_and_relogin = check_session_and_relogin_safe


import time
from urllib.parse import urljoin

def start_session_fresh(max_link_refresh=5):
    """Persona auth that NEVER blocks on a stale link.
    Each stale/expired inquiry (410/404) automatically triggers a brand-new
    /authentication POST -> new biometrics URL. Loop until authenticated."""
    s_ = ace.SingleSession()
    s_.auth = ace.get_credentials()
    for link_round in range(1, max_link_refresh + 1):
        r = s_.post(ace.brain_api_url + '/authentication')
        if r.status_code != 401:
            if link_round > 1:
                print('authenticated on a refreshed inquiry link.')
            return s_
        if 'WWW-Authenticate' not in r.headers or r.headers['WWW-Authenticate'] != 'persona':
            raise SystemExit(f'Authentication failed with HTTP {r.status_code} '
                             f'(not a persona challenge). Check credentials.')
        persona_url = urljoin(r.url, r.headers['Location'])
        print(f'\n--- biometrics link (attempt {link_round}/{max_link_refresh}) ---')
        print('1) Open this URL and complete biometrics:')
        print('   ' + persona_url)
        input('2) Press Enter here when done > ')
        for attempt in range(1, 4):
            rr = s_.post(persona_url)
            print(f'   auth check: HTTP {rr.status_code}', flush=True)
            if rr.status_code // 100 == 2:
                return s_
            if rr.status_code in (410, 404, 401):
                print('   link STALE/EXPIRED -> requesting a FRESH link automatically...')
                break                       # go around: new /authentication POST
            time.sleep(5)
        else:
            print('   not completed in 3 checks -> requesting a fresh link...')
    raise SystemExit('Could not complete biometrics after several fresh links. '
                     'Re-run the Setup cell.')

ace.start_session = start_session_fresh

ace.start_session = start_session_fresh

print('Connecting to WorldQuant BRAIN...')
s = ace.start_session()
s.request = functools.partial(s.request, timeout=300)  # anti-hang on every API call
print('Ready. Session expires in ~', ace.check_session_timeout(s), 's')
from datetime import datetime
# Get current date and time
now = datetime.now()
# Format as Hour:Minute:Second
current_time = now.strftime("%H:%M:%S")
print("Login at :", current_time)

Connecting to WorldQuant BRAIN...

--- biometrics link (attempt 1/5) ---
1) Open this URL and complete biometrics:
   https://api.worldquantbrain.com/authentication/persona?inquiry=inq_AUAgjsUQWb4rFxYXCDZzoMsXArZryN
   auth check: HTTP 201
Ready. Session expires in ~ 14398.755038 s
Login at : 07:36:03


In [18]:
# 1) env var -> 2) saved file -> 3) prompt-and-save (only happens once)
key = os.environ.get('OPENROUTER_API_KEY')
if not key:
    key_file = Path.home() / '.secrets' / 'openrouter.json'
    if key_file.exists():
        key = json.loads(key_file.read_text()).get('api_key', '')
    if not key:
        key = getpass.getpass('OpenRouter API key: ').strip()
        key_file.parent.mkdir(parents=True, exist_ok=True)
        key_file.write_text(json.dumps({'api_key': key}))
        print('Saved key to', key_file)
    os.environ['OPENROUTER_API_KEY'] = key

llm = ChatOpenAI(
    model='inclusionai/ling-3.0-flash',
    openai_api_key=os.environ['OPENROUTER_API_KEY'],
    openai_api_base='https://openrouter.ai/api/v1',
    temperature=0.5,
    timeout=180,
    model_kwargs={
        "reasoning": {"effort": "medium", "exclude": True},
        "response_format": {"type": "json_object"},
    },
)

/home/curo/miniforge3/envs/wq/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3762: UserWarning: Parameters {'reasoning'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  if await self.run_code(code, result, async_=asy):


In [19]:
# ============================================================
# HARDENED PATCHES — POLLER + FETCH (idempotent, one cell)
# Safe to re-run any number of times: patched functions are
# tagged and re-running this cell is a no-op.
# ============================================================
import time
import requests
import ace_lib as ace

_FETCH_RETRIES = 4
_FETCH_BACKOFF = 10
_NET_ERRORS    = (requests.exceptions.Timeout,
                  requests.exceptions.ConnectionError)


def _is_patched(fn):
    return getattr(fn, '_hardened', False)


def _wrap(module, name, wrapper_factory):
    """Wrap module.name exactly once. Idempotent by tag."""
    cur = getattr(module, name)
    if _is_patched(cur):
        print(f'skip: {name} already patched')
        return
    wrapped = wrapper_factory(cur)          # factory receives the TRUE original
    wrapped._hardened = True
    setattr(module, name, wrapped)
    print(f'patched: {name}')


def _get_retry(sess, url, per_call_timeout=120):
    last = None
    for attempt in range(1, _FETCH_RETRIES + 1):
        try:
            return sess.get(url, timeout=per_call_timeout)
        except _NET_ERRORS as e:
            last = e
            print(f"[fetch] {url.split('/')[-1]} attempt {attempt}/{_FETCH_RETRIES} "
                  f"network error: {e!r} -> retry in {_FETCH_BACKOFF}s", flush=True)
            if attempt < _FETCH_RETRIES:
                time.sleep(_FETCH_BACKOFF)
    raise last


def _failed_stats(simulate_data):
    return {"alpha_id": None, "simulate_data": simulate_data,
            "is_stats": None, "pnl": None, "stats": None,
            "is_tests": None, "train": None, "test": None}


# ---------- 1. multisimulation_progress (poller, hardened) ----------
def _make_poller(orig):
    def multisimulation_progress_v2(s_, simulate_response,
                                    max_wait_sec=3600, request_timeout=60):
        t0 = time.time()
        if simulate_response.status_code // 100 != 2:
            print(f"[sim] START FAILED  HTTP {simulate_response.status_code}: "
                  f"{simulate_response.text[:300]}", flush=True)
            return {"completed": False, "result": []}

        url = simulate_response.headers["Location"]
        poll, body = 0, {}
        while True:
            if time.time() - t0 > max_wait_sec:
                print(f"[sim] GIVE UP after {time.time()-t0:.0f}s. Batch URL: {url}", flush=True)
                return {"completed": False, "result": []}
            poll += 1
            print(f"[sim] poll {poll} | {time.time()-t0:.0f}s | waiting...", flush=True)

            if poll % 10 == 0:
                try:
                    if ace.check_session_timeout(s_) < 2000:
                        print("[sim] SESSION EXPIRED -> aborting batch.", flush=True)
                        return {"completed": False, "result": []}
                except Exception:
                    pass

            try:
                resp = s_.get(url, timeout=request_timeout)
            except Exception as e:
                print(f"[sim]   request error: {e!r} -> retry in 15s", flush=True)
                time.sleep(15)
                continue

            if resp.status_code // 100 != 2:
                ra = resp.headers.get("Retry-After")
                if ra is None:
                    print(f"[sim] HTTP {resp.status_code}, no Retry-After -> GIVING UP.", flush=True)
                    return {"completed": False, "result": []}
                wait = float(ra)
                if time.time() - t0 + wait > max_wait_sec:
                    print(f"[sim] GIVE UP (cap). Batch URL: {url}", flush=True)
                    return {"completed": False, "result": []}
                time.sleep(wait)
                continue

            try:
                body = resp.json()
            except Exception:
                print(f"[sim] unparseable body -> GIVING UP. {resp.text[:300]}", flush=True)
                return {"completed": False, "result": []}

            if body.get("status") == "ERROR":
                print("[sim] BATCH ERROR -> salvaging children individually...", flush=True)
                break
            if resp.headers.get("Retry-After", "0") == "0":
                break
            wait = float(resp.headers["Retry-After"])
            if time.time() - t0 + wait > max_wait_sec:
                print(f"[sim] GIVE UP (cap). Batch URL: {url}", flush=True)
                return {"completed": False, "result": []}
            time.sleep(wait)

        children = body.get("children", [])
        print(f"[sim] batch done | {len(children)} children | fetching results...", flush=True)
        results = []
        for i, child in enumerate(children, 1):
            alpha = None
            for _ in range(60):
                try:
                    c = s_.get(ace.brain_api_url + "/simulations/" + child,
                               timeout=request_timeout)
                except Exception as e:
                    print(f"[sim]   child {i} request error: {e!r} (retry)", flush=True)
                    time.sleep(10)
                    continue
                if c.headers.get("Retry-After"):
                    time.sleep(float(c.headers["Retry-After"]))
                    continue
                if c.status_code == 200:
                    alpha = c.json().get("alpha")
                break
            if not alpha:
                print(f"[sim]   child {i}/{len(children)} FAILED - skipping", flush=True)
                continue
            try:
                result = ace.get_simulation_result_json(s_, alpha)
            except Exception as e:
                print(f"[sim]   child {i} result fetch error: {e!r} -> skipping", flush=True)
                continue
            if result:
                results.append(result)
                print(f"[sim]   child {i}/{len(children)} OK -> alpha {alpha}", flush=True)

        print(f"[sim] done: {len(results)}/{len(children)} alphas returned", flush=True)
        return {"completed": len(results) > 0, "result": results}
    return multisimulation_progress_v2


# ---------- 2. get_simulation_result_json (bounded retries) ----------
def _make_simres(orig):
    def get_simulation_result_json_v2(s_, alpha_id):
        if alpha_id is None:
            return {}
        result = None
        for _i in range(90):                       # cap the original while-True
            result = _get_retry(s_, ace.brain_api_url + "/alphas/" + alpha_id)
            ra = result.headers.get("retry-after") or result.headers.get("Retry-After")
            if ra:
                time.sleep(float(ra))
                continue
            break
        try:
            return result.json()
        except Exception:
            ace.logger.error(f"alpha_id {alpha_id}, {result.headers}, "
                             f"{result.text}, {result.status_code}")
            return {}
    return get_simulation_result_json_v2


# ---------- 3. get_specified_alpha_stats (per-alpha isolation) ----------
def _make_stats(orig):
    def get_specified_alpha_stats_v2(s_, alpha_id, simulate_data, **cfg):
        for attempt in range(1, _FETCH_RETRIES + 1):
            try:
                return orig(s_, alpha_id, simulate_data, **cfg)
            except _NET_ERRORS as e:
                print(f"[stats] {alpha_id} attempt {attempt}/{_FETCH_RETRIES} failed: {e!r}",
                      flush=True)
                if attempt >= _FETCH_RETRIES:
                    return _failed_stats(simulate_data)
                time.sleep(_FETCH_BACKOFF)
        return _failed_stats(simulate_data)
    return get_specified_alpha_stats_v2


# ---------- 4. simulate_alpha_list_multi (batch-level guard) ----------
def _make_multi(orig):
    def simulate_alpha_list_multi_v2(s_, alpha_list, *a, **kw):
        try:
            return orig(s_, alpha_list, *a, **kw)
        except _NET_ERRORS as e:
            print(f"[multi-sim] batch aborted by network error: {e!r}\n"
                  f"[multi-sim] marking {len(alpha_list)} payload(s) as FAILED "
                  f"(alphas may exist on the platform - check manually)", flush=True)
            return [{"alpha_id": None, "simulate_data": x} for x in alpha_list]
    return simulate_alpha_list_multi_v2


# ---------- install (each wrap happens AT MOST ONCE, ever) ----------
_wrap(ace, 'multisimulation_progress',   _make_poller)
_wrap(ace, 'get_simulation_result_json', _make_simres)
_wrap(ace, 'get_specified_alpha_stats',  _make_stats)
_wrap(ace, 'simulate_alpha_list_multi',  _make_multi)

print("All HARDENED patches installed (idempotent - re-running this cell is safe).")

skip: multisimulation_progress already patched
skip: get_simulation_result_json already patched
skip: get_specified_alpha_stats already patched
skip: simulate_alpha_list_multi already patched
All HARDENED patches installed (idempotent - re-running this cell is safe).


In [41]:
# ============================================================================
# CATALOG CELL — dataset-scoped datafield caller
# Produces: ds, fields, field_types, field_desc, field_base, field_crowd,
#           DATASET_NAME, DATASET_DESC, DATASET_CAT, ops, DATASET
# Caches live in ./cache/ ; legacy flat caches in the notebook root are
# auto-migrated on first run (idempotent - safe to re-run).
#
# CACHE FIX: caches now carry their (dataset_id, region, universe, delay, type)
# as metadata INSIDE the pickle, so validity is checked against real context,
# NOT by guessing from field names (which broke for generic ids like
# fundamental6 whose fields are `assets`, `cash`, ...).
# ============================================================================
import helpful_functions as hf

# ----------------------------- CONFIG ---------------------------------------
REGION, UNIVERSE, DELAY, DECAY = "ASI", "TOP500", 1, 2
NEU            = "REVERSION_AND_MOMENTUM"   # neutralization: SIMULATION SETTING level
MAX_TRADE      = "OFF"                      # simulation setting
SIMULATION_MODE = "QUICK"                   # "QUICK"=screening speed, "FULL"=exact
TYPE           = "MATRIX"                   # "MATRIX"/"VECTOR"/None(all)
NAME           = None
DATASET_ID     = "fundamental17"
BACKFILL_D     = 100
ALPHACOUNT_MAX = 999999999                  # crowding-aware field filter (50 = strict)
FORCE_CRAWL    = False                      # True -> ignore catalog cache
CORRELATION_FILTER = False   # True = measure prod-corr for top-K + enforce < 0.7 selection
                            # False = pure fitness discovery (no prod-corr API calls)

# --------------------- 0. OPERATOR BLOCKLIST (baked into CATALOG) ------------
# DECAY=4 at settings level handles position smoothing -> the LLM gets NO
# turnover-control operators. Signal speed is controlled via lookback windows.
# Also blocks control-flow/tuple-breaking ops the model repeatedly misuses.
OPS_BLOCKLIST = {
    'ts_target_tvr_decay', 'ts_target_tvr_hump', 'ts_target_tvr_delta_limit',
    'ts_delta_limit', 'inst_tvr',
    'hump', 'hump_decay', 'jump_decay', 'ts_weighted_decay',
    'ts_decay_linear', 'ts_decay_exp_window',
    'ts_step',
    'if_else', 'gt', 'lt', 'greater', 'less', 'winsorize',
}

# --------------------- 1. resolve dataset id --------------------------------
def resolve_dataset_id(query):
    """NAME substring (or id substring) → dataset id."""
    datasets_df = ace.get_datasets(s, "EQUITY", region=REGION, delay=DELAY, universe=UNIVERSE)
    sel = datasets_df[
        (datasets_df["delay"]    == DELAY) &
        (datasets_df["region"]   == REGION) &
        (datasets_df["universe"] == UNIVERSE) &
        (datasets_df["id"].str.contains(query, case=False, na=False) |
         datasets_df["name"].str.contains(query, case=False, na=False))
    ]
    if sel.empty:
        print("No match. Available dataset ids:")
        print(sorted(datasets_df["id"].dropna().unique().tolist()))
        raise ValueError("Adjust NAME or set DATASET_ID, then re-run.")
    print("matches:", sel.id.values.tolist())
    return sel.id.values.tolist()[0]

dataset_id = DATASET_ID or resolve_dataset_id(NAME)
print("dataset id :", dataset_id)

# --------------------- 1b. DATASET METADATA (NON-FATAL) ----------------------
DATASET_NAME, DATASET_DESC, DATASET_CAT = dataset_id, '', ''
try:
    _dsets = ace.get_datasets(s, region=REGION, universe=UNIVERSE, delay=DELAY)
except Exception as _e:
    print(f"WARNING: get_datasets metadata TIMED OUT / failed ({_e!r}) - using placeholder", flush=True)
else:
    _drow = _dsets[_dsets['id'] == dataset_id]
    if _drow.empty:
        print(f"WARNING: dataset '{dataset_id}' not found in /data-sets metadata", flush=True)
    else:
        _m = _drow.iloc[0]
        DATASET_NAME = _m.get('name') or dataset_id
        DATASET_DESC = str(_m.get('description') or '')[:300]
        DATASET_CAT  = str(_m.get('category_name') or _m.get('category') or '')[:60]
print(f"Dataset metadata: '{DATASET_NAME}' | cat: {DATASET_CAT!r}")

# --------------------- 2. crawl ALL datafields ------------------------------
def fetch_dataset_fields(dataset_id, data_type=None):
    """ALL datafields of one dataset via direct API (ace_lib-independent).
    Paginates by the server-reported 'count' (NOT by short-page detection)."""
    rows, offset, limit = [], 0, 50
    total = None
    while True:
        url = (f"{ace.brain_api_url}/data-fields?instrumentType=EQUITY"
               f"&region={REGION}&delay={DELAY}&universe={UNIVERSE}"
               f"&dataset.id={dataset_id}&limit={limit}&offset={offset}")
        r = s.get(url)
        try:
            ace._check_rate_limit(r)
        except AttributeError:
            pass
        if r.status_code != 200:
            raise ValueError(f"API error {r.status_code}: {r.text[:200]}")
        j = r.json()
        batch = j.get("results") or []
        if total is None:
            total = j.get("count") or (offset + len(batch))
        rows.extend(batch)
        offset += len(batch)
        print(f"\r  fetched {len(rows)}/{total} fields...", end="")
        if total is not None and offset >= total:
            break
        if not batch:              # safety: no rows returned -> stop
            break
    print()

    df = pd.DataFrame(rows)
    if not df.empty:
        df = hf.expand_dict_columns(df)
    if data_type and "type" in df.columns:
        df = df[df["type"] == data_type]
    return df.drop_duplicates(subset="id").reset_index(drop=True)

# ---------------- CACHE: metadata-verified + empty-safe ----------------------
import os as _os
import shutil as _shutil
from pathlib import Path as _Path

_CACHE_DIR     = _Path('cache')
_CACHE_DIR.mkdir(exist_ok=True)                      # fail loud on permission errors
_ARTIFACT_FILE = _CACHE_DIR / 'catalog_cache.pkl'    # legacy unkeyed cache (foldered name)
CACHE_FILE     = _CACHE_DIR / f"catalog_cache_{REGION}_{UNIVERSE}_D{DELAY}_{dataset_id}.pkl"

# ALL legacy flat caches still sitting in the notebook root (keyed + unkeyed)
_LEGACY_GLOB = list(_Path('.').glob('catalog_cache*.pkl'))


def _read_cache(path):
    """Return (df, meta). New format = dict with metadata; legacy = raw DataFrame."""
    try:
        obj = pd.read_pickle(path)
    except Exception as _e:
        print(f'WARNING: unreadable cache {path}: {_e!r}', flush=True)
        return None, None
    if isinstance(obj, dict) and 'df' in obj:
        return obj['df'], obj
    if isinstance(obj, pd.DataFrame):
        return obj, None                # legacy cache: no metadata attached
    return None, None


def _cache_ok(df, meta):
    """Usable iff non-empty AND (when metadata exists) it matches this context.
    Legacy caches carry no metadata and are accepted if non-empty (the filename
    IS the key). This is what fixes the perpetual 'mismatched' re-crawl."""
    if df is None or df.empty:
        return False
    if meta is None:
        return True
    return (meta.get('dataset_id') == dataset_id
            and meta.get('region')   == REGION
            and meta.get('universe') == UNIVERSE
            and meta.get('delay')    == DELAY
            and meta.get('type')     == TYPE)


def _save_cache(df):
    """Persist the frame WITH its context so a stale/foreign cache can never
    masquerade as this one (field names are NOT a reliable dataset fingerprint)."""
    pd.to_pickle({'dataset_id': dataset_id, 'region': REGION, 'universe': UNIVERSE,
                  'delay': DELAY, 'type': TYPE, 'df': df}, CACHE_FILE)


# ---- one-time migration of ALL legacy flat caches (idempotent) --------------
for _legacy in _LEGACY_GLOB:
    _target = _CACHE_DIR / _legacy.name
    if _target.exists():
        _legacy.unlink()                             # already migrated - clean up duplicate
        print(f'[cache] removed duplicate legacy {_legacy} (migrated copy exists)')
        continue
    try:
        _shutil.move(str(_legacy), str(_target))
        print(f'[cache] migrated {_legacy} -> {_target}')
    except Exception as _e:
        print(f'WARNING: could not migrate {_legacy}: {_e!r} (continuing)', flush=True)

# fold the unkeyed cache into the keyed name if it belongs to THIS dataset
if _ARTIFACT_FILE.exists() and not CACHE_FILE.exists():
    _df, _meta = _read_cache(_ARTIFACT_FILE)
    if _df is not None and not _df.empty:
        _df.to_pickle(CACHE_FILE)                    # save as legacy frame; _cache_ok accepts it
        print(f'folded legacy unkeyed cache -> {CACHE_FILE} ({len(_df)} fields)')
    _ARTIFACT_FILE.unlink()                          # consumed either way

# ---- load or crawl ----------------------------------------------------------
ds = None
if CACHE_FILE.exists() and not FORCE_CRAWL:
    _df, _meta = _read_cache(CACHE_FILE)
    if _cache_ok(_df, _meta):
        ds = _df
        print(f'LOADED cached datafields ({len(ds)}) for {dataset_id} '
              f'[{REGION}/{UNIVERSE}/D{DELAY}] from {CACHE_FILE} '
              f'- set FORCE_CRAWL=True to re-fetch')
    else:
        print(f'WARNING: cache {CACHE_FILE} is empty/foreign ({0 if _df is None else len(_df)} rows) '
              f'-> invalidating and re-crawling', flush=True)
        CACHE_FILE.unlink()

if ds is None:
    ds = fetch_dataset_fields(dataset_id, data_type=TYPE)
    if ds.empty:
        print(f'WARNING: crawl returned 0 fields for {dataset_id} with '
              f'TYPE={TYPE!r} - NOT caching empty result', flush=True)
    else:
        _save_cache(ds)
        print(f'CRAWLED fresh datafields ({len(ds)}) -> {CACHE_FILE}')

# --------------------- 3. ALL FILTERING FIRST (before building vars) ----------
if "alphaCount" in ds.columns:
    n_before = len(ds)
    ds = ds[pd.to_numeric(ds["alphaCount"], errors="coerce").fillna(0) <= ALPHACOUNT_MAX]
    print(f"crowding filter: alphaCount <= {ALPHACOUNT_MAX} -> "
          f"{len(ds)} of {n_before} datafields satisfy the condition")
else:
    print("WARNING: no alphaCount column -> crowding filter skipped")

ds = ds.sort_values("coverage", ascending=False, na_position="last").reset_index(drop=True)

# --------------------- 4. variables the agent consumes (from FILTERED ds) ----
field_types = dict(zip(ds["id"], ds["type"]))
field_desc  = dict(zip(ds["id"], ds.get("description", "").fillna("")))
field_crowd = (dict(zip(ds["id"], pd.to_numeric(ds["alphaCount"], errors="coerce").fillna(0)))
               if "alphaCount" in ds.columns else {})

def entry_expr(fid):
    """MATRIX : ts_backfill(field, 100);  VECTOR : ts_backfill(vec_avg(field), 100)"""
    return (f"ts_backfill({fid}, {BACKFILL_D})" if field_types[fid] == "MATRIX"
            else f"ts_backfill(vec_avg({fid}), {BACKFILL_D})")

field_base = {fid: entry_expr(fid) for fid in ds["id"]}

# --------------------- 4b. SUBSET SELECTION ----------------------------------
#fields = ds["id"].tolist()                                  # ← default: all
# fields = ds.iloc[0:40]["id"].tolist()                     # top 40 by coverage
#fields = fields[:40]                                      # first 40 (id order)
fields = ds[ds["coverage"] >= 0.4]["id"].tolist()
#fields = fields[40:80]
fields = fields[20:40]
missing = [f for f in fields if f not in field_types]
if missing:
    raise ValueError(f"Fields not in dataset {dataset_id}: {missing}")

DATASET = dataset_id
history = []                       # full forget on dataset switch
_corr_cache = {}                   # forget cached prod-corr too
print('history wiped for new dataset')

# --------------------- 5. operators for the prompt --------------------------
ops_df = ace.get_operators(s)
ops = sorted(set(o for o in ops_df[ops_df["scope"] == "REGULAR"]["name"].unique()) - OPS_BLOCKLIST)
print(f"operators: {len(ops)} after blocklist")

# --------------------- summary ----------------------------------------------
show = [c for c in ["id", "type", "coverage", "userCount", "description"] if c in ds.columns]
display(ds[show])
print("success\ndataset id :", dataset_id, "\ncount :", len(ds),
      "\nagent field pool :", len(fields), "\nREGULAR operators :", len(ops))
print(f"\nsample ENTRY expression ({fields[0]}):")
print(" ", field_base[fields[0]])

dataset id : fundamental17
Dataset metadata: 'Direct Fundamental Data' | cat: 'Fundamental'
LOADED cached datafields (370) for fundamental17 [ASI/TOP500/D1] from cache/catalog_cache_ASI_TOP500_D1_fundamental17.pkl - set FORCE_CRAWL=True to re-fetch
crowding filter: alphaCount <= 999999999 -> 370 of 370 datafields satisfy the condition
history wiped for new dataset
operators: 72 after blocklist


,id,type,coverage,userCount,description
0,usd_to_reporting_fx_rate_2,MATRIX,0.8467,0,USD to reporting currency exchange rate
1,reporting_to_pricing_currency_fx_rate,MATRIX,0.8467,0,Exchange rate from reporting currency to the b...
2,usd_to_reporting_currency_fx_rate,MATRIX,0.8467,0,The exchange rate used to convert US dollars t...
3,reporting_to_pricing_currency_fx,MATRIX,0.8467,0,Exchange rate from the company’s reporting cur...
4,financial_reporting_currency_code_3,MATRIX,0.8467,0,Reporting currency used in the company’s finan...
...,...,...,...,...,...
365,fnd17_ptmintcov,MATRIX,0.4305,2,Interest coverage for the prior trailing 12 mo...
366,fnd17_divtrend10,MATRIX,0.4199,0,Ten-year compound annual growth rate of divide...
367,fnd17_pxedra,MATRIX,0.3737,1,Research and development expense for the most ...
368,fnd17_divnqpdt,MATRIX,0.3603,1,Dividend - next quarterly pay date


success
dataset id : fundamental17 
count : 370 
agent field pool : 20 
REGULAR operators : 72

sample ENTRY expression (reporting_currency_code_ras8):
  ts_backfill(reporting_currency_code_ras8, 100)


In [42]:
print('dataset_id :', dataset_id)
print('DATASET    :', DATASET)
print('n fields   :', len(ds))
print('FIELD PREFIX SAMPLE:', sorted(set(str(x).split('_')[0] for x in ds['id']))[:10])
if len(ds) and 'fnd86' in str(ds['id'].iloc[0]):
    print('>>> STILL FUNDAMENTAL86 - cache is stale, see Possibility 2 <<<')

dataset_id : fundamental17
DATASET    : fundamental17
n fields   : 370
FIELD PREFIX SAMPLE: ['enterprise', 'financial', 'fnd17', 'gross', 'pricing', 'quarterly', 'reporting', 'usd']


In [43]:
#GENERATOR CELL
import ast, json

def _content_to_text(content) -> str:
    """Flatten an LLM reply to plain text, tolerating None / str / block lists."""
    if content is None:
        return ''
    if isinstance(content, str):
        return content
    if isinstance(content, (list, tuple)):
        out = []
        for blk in content:
            if isinstance(blk, str):
                out.append(blk)
            elif isinstance(blk, dict):
                if blk.get('type') == 'text' and isinstance(blk.get('text'), str):
                    out.append(blk['text'])                 # standard text block
                elif isinstance(blk.get('content'), str):
                    out.append(blk['content'])              # nested string
                elif isinstance(blk.get('content'), (list, tuple)):
                    for sub in blk['content']:
                        if isinstance(sub, dict) and isinstance(sub.get('text'), str):
                            out.append(sub['text'])         # nested blocks
        return '\n'.join(out)
    return str(content)

def _llm_json_objects(content):
    """Return list[dict] parsed from an LLM JSON-array reply. [] if unusable."""
    text = (_content_to_text(content) or '').strip()
    text = text.replace('```json', '').replace('```', '').strip()
    if not text:
        return []
    try:
        raw = _parse_json_array(text)
        if isinstance(raw, dict) and 'proposals' in raw:
            raw = raw['proposals']
        return raw if isinstance(raw, list) else []
    except ValueError:
        return []
def _parse_json_array(text):
    """Robustly parse a JSON array from model output (clean / fenced / single-quoted)."""
    t = text.strip()
    try:
        return json.loads(t)
    except Exception:
        pass
    s, e = t.find('['), t.rfind(']')
    if s != -1 and e > s:
        cand = t[s:e + 1]
        try:
            return json.loads(cand)
        except Exception:
            pass
        try:
            return ast.literal_eval(cand)
        except Exception:
            pass
    raise ValueError(f'Could not parse JSON array from model output.\n'
                     f'First 500 chars: {t[:500]!r}')

def syntax_guide_text():
    """Render the official-doc syntax contract from OPS_RULES.
    One source of truth: prompt guide and validator share this table."""
    def _sig(pmin, pmax, kwargs):
        args = ['x'] * pmin
        s = f"{len(args)} positional" if pmax is None else (
            f"{len(args)} positional" if pmax == pmin else f"{pmin}-{pmax} positional")
        if kwargs:
            s += f" | kw-only: {', '.join(sorted(kwargs))}"
        return s
    lines = [f"  {op:<22s} {_sig(*rules)}" for op, rules in sorted(OPS_RULES.items())]
    return "## Operator Syntax Contract (from official BRAIN docs)\n" + \
           "Each op: positional arity | options that MUST be written as name=value.\n" + \
           "Global rules: options are KEYWORD-form only; lookbacks are BARE INTEGERS;\n" + \
           "no tuples, no infix operators; rank(x) takes EXACTLY ONE input.\n" + \
           "\n".join(lines)
def generate(history, n=10, top_k=15, recent_k=5):
    """One LLM call: proposals + rationale, with crowding-aware feedback.
    history is filtered to the CURRENT dataset context before ranking,
    so foreign-dataset alphas never leak into the prompt.
    Respects CORRELATION_FILTER: when False, crowding feedback/objective
    are dropped entirely (pure fitness discovery)."""
    # ---- CONTEXT FILTER: only this dataset+region+delay's entries ----
    ctx = f"{REGION}|{UNIVERSE}|D{DELAY}|dec{DECAY}|{NEU}|{DATASET}"
    ctx_hist = [r for r in history if r.get('ctx') == ctx]
    if not ctx_hist:
        ctx_hist = [r for r in history
                    if any(fid in str(r.get('expression', '')) for fid in fields)]
    ctx_hist = ctx_hist or history

    # ---- correlation filter ON/OFF (global from CATALOG config) ----
    filter_on = bool(globals().get('CORRELATION_FILTER', True))

    field_lines = '\n'.join(
        f"- ENTRY: {field_base[f]}\n"
        f"  ({field_types[f]}, prod alphas on field: {field_crowd.get(f, '?')}) "
        f"{str(field_desc.get(f, ''))[:250]}"
        for f in fields
    )

    def _line(i, r):
        pc = r.get('prod_corr')
        pc_tag = (f", prod-corr {pc:.2f}" + (" [CROWDED]" if pc >= PROD_CORR_LIMIT else "")
                  if pc is not None else "")
        rat = f" | was because: {r['rationale'][:70]}" if r.get('rationale') else ''
        return (f"{i+1}. {r['expression'][:85]} -> sharpe {r['sharpe']:.2f}, "
                f"fitness {r['fitness']:.2f}, turnover {r['turnover']:.2f}{pc_tag}{rat}"
                + ('  [SIMULATION FAILED]' if r['alpha_id'] is None else ''))

    # ---- SLIM HISTORY: best top_k + most recent recent_k, de-duplicated ----
    ranked = sorted(ctx_hist, key=lambda r: r['fitness'], reverse=True)
    selected = ranked[:top_k]
    seen = {r['expression'] for r in selected}
    for r in ctx_hist[-recent_k:]:
        if r['expression'] not in seen:
            selected.append(r)
            seen.add(r['expression'])
    hist = '\n'.join(_line(i, r) for i, r in enumerate(selected)) \
           or '(no history yet - explore freely)'

    # ---- CROWDING FEEDBACK: ONLY when correlation filter is ON ----
    crowd_block = ''
    if filter_on:
        crowded = sorted([r for r in ctx_hist
                          if r.get('prod_corr') is not None and r['prod_corr'] >= PROD_CORR_LIMIT],
                         key=lambda r: r['fitness'], reverse=True)
        if crowded:
            lines = '\n'.join(f"  - {r['expression'][:80]}  (fitness {r['fitness']:.2f}, "
                              f"prod-corr {r['prod_corr']:.2f})" for r in crowded[:5])
            crowd_block = f'''
## CROWDING FEEDBACK (CRITICAL - read before proposing)
These high-performing alphas are not accepted because of high correlation (>={PROD_CORR_LIMIT}) with PRODUCTION alphas:
{lines}
Their IDEAS are crowded, not their syntax. You MUST NOT re-propose variants of them
(different lookback, extra rank() wrapper, renamed field of same meaning).
To beat them, change the INFORMATION CONTENT: different/combined fields, add
conditioning, or a structurally different transform — anything that decorrelates
the PnL stream while keeping fitness.'''

    # ---- REFERENCE POINT + OBJECTIVE (flag-aware) ----
    try:
        _ub = best_uncrowded(history)
        if filter_on:
            _ref = (f"The best UNcrowded alpha so far has fitness {_ub['fitness']:.2f} "
                    f"(prod-corr {_ub.get('prod_corr') if _ub.get('prod_corr') is not None else 'unmeasured'}).")
        else:
            _ref = f"The best alpha so far has fitness {_ub['fitness']:.2f}."
    except Exception:
        _ref = "(no reference yet)"
    objective = (f"Propose {n} NEW expressions that beat the best fitness so far"
                 + (f" AND has low correlation (prod-corr < {PROD_CORR_LIMIT})." if filter_on else "."))

    prompt = f'''You are a quant researcher. Region {REGION}, Universe {UNIVERSE}.

## Dataset
Name    : {DATASET_NAME}   (id: {DATASET})
Category: {DATASET_CAT or "n/a"}
Description   : {DATASET_DESC or "No description available."}

Neutralization is applied at SIMULATION SETTINGS level. Do NOT neutralize
group exposures inside the expression (no group_rank / group_zscore used as
a neutralizer substitute).

## Datafields
Each field has a full wrapper expression below.

{field_lines}

## ENTRY RULES (critical — violations are auto-rejected)
- Copy the wrapper expression EXACTLY as written, nested inside your operators.
- Do NOT abbreviate it. Do NOT invent a short variable name for it.
- Do NOT write the word "entry" anywhere in your expression — it is not a
  variable, it is a placeholder in these instructions only.

  WRONG:   ts_zscore(entry_section_positive_score, 20)   <- invented variable
  WRONG:   ts_zscore(ENTRY, 20)                          <- placeholder, not a variable
  CORRECT: ts_zscore(ts_backfill(vec_avg(section_positive_score), 100), 20)

## Operators
{syntax_guide_text()}

## Syntax rules (each rule below has caused rejections in the past)
1. ts_* operators need a lookback: ts_zscore(..., 10), ts_delta(..., 5).
2. Lookbacks must be BARE integer literals. Never arithmetic or variables:
   ts_zscore(ENTRY, 20-5)  -> INVALID
   ts_zscore(ENTRY, 20)    -> VALID
3. Match the operator's exact arity. ts_corr needs TWO series + lookback:
   ts_corr(ENTRY_A, ENTRY_B, 60). One-input ops like ts_zscore take exactly
   one series argument, plus its lookback.
4. Vector fields MUST go through their wrapper (e.g. vec_avg(...)) before any
   time-series operator.
5. Parentheses balanced; the whole expression must be ONE expression,
   no comments, no newlines, no trailing semicolons.

## Your history (append-only scoreboard — read it before proposing)
{hist}

{crowd_block}

## Strategy for this round
- Improve on alphas with the HIGHEST FITNESS
- Use the maximum number of 3 datafields for one alpha expression
- Propose DIVERSE ideas: vary the transformation family (momentum/reversal,
  rank/normalize, ts_corr/interaction, group-relative signal, volatility
  conditioning) — at least 3 distinct families across the batch.
- Keep expressions readable: deep nesting is allowed but prefer <= 3 levels.
- Reference: {_ref}

## Objective
{objective}

## Output format (STRICT)
For EVERY expression give a 1-2 sentence rationale: the signal/edge exploited,
which wrapper field is used, and why it should beat the previous best without
crowding.

Return ONLY a raw JSON object — no markdown fences, no prose before or after:

{{"proposals": [
    {{"expression": "...", "rationale": "..."}},
    {{"expression": "...", "rationale": "..."}}
]}}
'''

    resp = llm.invoke([
        SystemMessage(content='Return only a raw JSON array of objects with keys "expression" and "rationale". No commentary.'),
        HumanMessage(content=prompt),
    ])

    text = _content_to_text(resp.content).strip().replace('```json', '').replace('```', '').strip()
    raw = _parse_json_array(text)
    proposals = []
    for item in raw[:n]:
        if isinstance(item, dict):
            proposals.append({'expression': str(item.get('expression', '')).strip(),
                              'rationale': str(item.get('rationale', '')).strip()})
        elif isinstance(item, str):
            proposals.append({'expression': item.strip(), 'rationale': '(no rationale given)'})
    return [p for p in proposals if p['expression']]

In [44]:
# ============================================================================
# CROWDING MEASUREMENT - prod-corr for top-K of every iteration (cached)
# Respects CORRELATION_FILTER:
#   True  -> measure prod-corr for top-K, selection prefers uncrowded (< 0.7)
#   False -> pure fitness discovery: no prod-corr queries, no constraint
# FIX: best_uncrowded() now returns None when there is NO eligible alpha
#      (empty batch / all-failed iteration) instead of raising max([]).
# ============================================================================
import time

PROD_CORR_LIMIT = 0.7      # platform submission gate
CORR_TOP_K      = 3        # measure top-3 of each batch (endpoint is slow)

if '_corr_cache' not in globals():
    _corr_cache = {}       # alpha_id -> prod corr (measured ONCE per alpha)


def _filter_on():
    return bool(globals().get('CORRELATION_FILTER', True))


def prod_corr_of(alpha_id, _retries=2, _backoff=15):
    """Max prod-corr, same logic as the platform's own PROD_CORRELATION check.
    0.0 = no production alphas correlated above zero.
    None = endpoint failed after retries -> treated as UNMEASURED (loud warning)."""
    if alpha_id in _corr_cache:
        return _corr_cache[alpha_id]
    for attempt in range(1, _retries + 2):
        try:
            pc = ace.get_prod_corr(s, alpha_id)
            if not pc.empty and 'alphas' in pc.columns and (pc['alphas'] > 0).any():
                _corr_cache[alpha_id] = float(pc.loc[pc['alphas'] > 0, 'max'].max())
            else:
                _corr_cache[alpha_id] = 0.0
            return _corr_cache[alpha_id]
        except Exception as e:
            print(f'[prod-corr] {alpha_id} attempt {attempt}/{_retries + 1} failed: {e!r}',
                  flush=True)
            if attempt <= _retries:
                time.sleep(_backoff)
    print(f'[prod-corr] {alpha_id} UNMEASURED (endpoint down) -> skipped, not marked crowded',
          flush=True)
    _corr_cache[alpha_id] = None
    return None


def measure_topk(results, k=CORR_TOP_K):
    """Attach prod_corr to the k best NEW results of this batch (in place).
    CORRELATION_FILTER=False -> returns immediately, zero prod-corr API calls."""
    if not _filter_on():
        return
    n = 0
    for r in sorted((r for r in results if r.get('alpha_id')),
                    key=lambda r: r['fitness'], reverse=True):
        if n >= k:
            break
        pc = prod_corr_of(r['alpha_id'])
        if pc is not None:
            r['prod_corr'] = pc
        n += 1


def best_uncrowded(history, limit=PROD_CORR_LIMIT):
    """Best by fitness, BUT only among alphas from the CURRENT dataset context.
    - CORRELATION_FILTER=True:  best among prod-corr < limit (or unmeasured)
    - CORRELATION_FILTER=False: outright best by fitness
    Foreign-dataset rows are NEVER considered - the agent forgets them.
    Returns None when no successful alpha exists yet (empty / all-failed batch)."""
    ctx  = f"{REGION}|{UNIVERSE}|D{DELAY}|dec{DECAY}|{NEU}|{DATASET}"
    _ffs = fields or []
    mine = [r for r in history
            if r.get('alpha_id') is not None
            and (r.get('ctx') == ctx or any(f in str(r.get('expression', '')) for f in _ffs))]
    if not mine:                                        # empty current ctx
        mine = [r for r in history if r.get('alpha_id') is not None]
    if not mine:                                        # <-- THE FIX
        return None                                     # nothing to rank yet
    if not _filter_on():
        return max(mine, key=lambda r: r['fitness'])
    eligible = [r for r in mine if (r.get('prod_corr') is None or r['prod_corr'] < limit)]
    return max(eligible or mine, key=lambda r: r['fitness'])

In [45]:
# ============================================================================
# FAILED-SIM RETRY QUEUE — resimulate failed child alphas in the NEXT iteration
# A multi-sim batch can flip to ERROR (one poison child kills the batch) or
# individual children can fail. Those expressions are queued and re-simulated
# in later iterations, ONE AT A TIME, so a poison retry can never take down
# fresh proposals. Bounded attempts -> no infinite loop.
# State lives in history/retry_<ctx>.json (atomic writes; survives restarts).
# ============================================================================
import json, time
from pathlib import Path

RETRY_MAX_ATTEMPTS = 1   # max TOTAL simulations per expression (1 fresh + 1 retry)
RETRY_PER_ITER     = 1   # how many queued retries to attempt each iteration

_HISTORY_DIR_R = Path('history')
_HISTORY_DIR_R.mkdir(exist_ok=True)          # fail loud on permission errors

_retry_queue = {}
_retry_ctx   = None


def _retry_ctx_now():
    return f"{REGION}|{UNIVERSE}|D{DELAY}|dec{DECAY}|{NEU}|{DATASET}"


def _retry_path():
    return _HISTORY_DIR_R / f"retry_{REGION}_{UNIVERSE}_D{DELAY}_{DATASET}.json"


def _ensure_retry_loaded():
    """(Re)load the queue whenever the dataset context changes."""
    global _retry_queue, _retry_ctx
    if _retry_ctx == _retry_ctx_now():
        return
    p = _retry_path()
    if p.exists():
        try:
            _retry_queue = json.loads(p.read_text())
            print(f'[retry] loaded {len(_retry_queue)} queued failure(s) from {p}', flush=True)
        except Exception as e:
            print(f'WARNING: unreadable {p} ({e!r}) -> empty queue', flush=True)
            _retry_queue = {}
    else:
        _retry_queue = {}
    _retry_ctx = _retry_ctx_now()


def _save_retry():
    """Atomic save so a crash mid-write can never corrupt the queue."""
    p = _retry_path()
    tmp = p.with_suffix('.tmp')
    tmp.write_text(json.dumps(_retry_queue, indent=1))
    tmp.replace(p)


def enqueue_failed(results):
    """Success -> drop from queue (dedupe). Failure -> (re)queue with attempt++.
    Expressions that reach RETRY_MAX_ATTEMPTS are dropped LOUDLY."""
    _ensure_retry_loaded()
    changed = False
    for r in results:
        expr = (r.get('expression') or '').strip()
        if not expr:
            continue
        if r.get('alpha_id'):                     # simulated OK -> done with it
            if expr in _retry_queue:
                del _retry_queue[expr]
                changed = True
            continue
        ent = _retry_queue.get(expr, {'rationale': r.get('rationale', ''), 'attempts': 0})
        ent['attempts']  = int(ent.get('attempts', 0)) + 1
        ent['rationale'] = ent.get('rationale') or r.get('rationale', '')
        ent['last']      = time.strftime('%Y-%m-%d %H:%M:%S')
        _retry_queue[expr] = ent
        changed = True

    dead = [e for e, v in _retry_queue.items()
            if int(v.get('attempts', 1)) >= RETRY_MAX_ATTEMPTS]
    for e in dead:
        print(f'[retry] GIVING UP after {_retry_queue[e]["attempts"]} attempt(s): {e[:80]}',
              flush=True)
        del _retry_queue[e]
        changed = True

    if changed:
        _save_retry()


def run_retries_isolated(n=RETRY_PER_ITER):
    """Pop up to n queued expressions and simulate EACH as its own SINGLE sim
    (len==1 -> ace takes the single-alpha path). Isolation means a poison retry
    cannot flip a fresh or sibling batch to ERROR. Returns result rows."""
    _ensure_retry_loaded()
    if not _retry_queue or n <= 0:
        return []
    picked = list(_retry_queue.items())[:n]
    print(f'[retry] attempting {len(picked)} of {len(_retry_queue)} queued failure(s), isolated:',
          flush=True)
    out = []
    for expr, v in picked:
        print(f'  - retry (attempt {int(v.get("attempts", 0)) + 1}) {expr[:80]}', flush=True)
        try:
            out.extend(simulate_and_score([{'expression': expr,
                                            'rationale': v.get('rationale', '(retry)')}]))
        except Exception as e:                         # fail LOUD, don't kill loop
            print(f'  [retry] unexpected error on {expr[:60]}: {e!r} -> leaving in queue',
                  flush=True)
    return out


def retry_queue_size():
    _ensure_retry_loaded()
    return len(_retry_queue)

In [46]:
# ============================================================================
# LOCAL VALIDATOR v7 — doc-verified operator rules + whitespace tolerance
# + auto_repair as a bounded FIXPOINT (re-validate after every pass)
# Pipeline: repair_entries -> auto_repair (fixpoint) -> validate
# Unrepairable proposals go to LLM refill; refill failures are non-fatal.
# ============================================================================
import ast, json, re

for _name in ('_content_to_text', 'ops', 'ops_df', 'fields', 'field_base',
              'field_types', 'BACKFILL_D'):
    if _name not in globals():
        raise NameError(f"'{_name}' is missing -> re-run the GENERATOR/CATALOG cells first.")

# ------------------------------------------------------------------
# OPS_RULES:  op -> (min_positional, max_positional|None=variadic,
#                    {keyword-only params})
# Source: official BRAIN operator docs (platform /learn/operators)
# ------------------------------------------------------------------
TRUE_ARITH = {'add', 'subtract', 'multiply', 'divide', 'min', 'max',
              'power', 'signed_power'}   # positional floats legit here

OPS_RULES = {
    # --- arithmetic ---
    'abs': (1, 1, {}), 'inverse': (1, 1, {}), 'log': (1, 1, {}),
    'reverse': (1, 1, {}), 'sign': (1, 1, {}), 'sqrt': (1, 1, {}),
    'densify': (1, 1, {}),
    'add': (2, None, {'filter'}), 'multiply': (2, None, {'filter'}),
    'subtract': (2, None, {'filter'}),
    'divide': (2, 2, {}), 'power': (2, 2, {}), 'signed_power': (2, 2, {}),
    'max': (2, None, {}), 'min': (2, None, {}),
    'to_nan': (1, 1, {'value', 'reverse'}),
    # --- logical ---
    'and': (2, 2, {}), 'or': (2, 2, {}), 'not': (1, 1, {}),
    'is_nan': (1, 1, {}),
    # --- time series ---
    'days_from_last_change': (1, 1, {}),
    'kth_element': (2, 3, {'ignore'}),
    'last_diff_value': (2, 2, {}),
    'ts_arg_max': (2, 2, {}), 'ts_arg_min': (2, 2, {}),
    'ts_std_dev': (2, 2, {}),
    'ts_backfill': (1, 2, {'k'}),
    'ts_corr': (3, 3, {}), 'ts_covariance': (3, 3, {}),
    'ts_count_nans': (2, 2, {}), 'ts_delay': (2, 2, {}),
    'ts_delta': (2, 2, {}), 'ts_max': (2, 2, {}), 'ts_min': (2, 2, {}),
    'ts_mean': (2, 2, {}), 'ts_product': (2, 2, {}),
    'ts_av_diff': (2, 2, {}), 'ts_sum': (2, 2, {}),
    'ts_rank': (2, 2, {'constant'}), 'ts_scale': (2, 2, {'constant'}),
    'ts_regression': (3, 3, {'lag', 'rettype'}),
    'ts_quantile': (2, 2, {'driver'}),
    'ts_zscore': (2, 2, {}),
    # --- cross sectional ---
    'rank': (1, 1, {'rate'}),
    'zscore': (1, 1, {}),
    'normalize': (1, 1, {'useStd', 'limit'}),
    'quantile': (1, 1, {'driver', 'sigma'}),
    'scale': (1, 1, {'scale', 'longscale', 'shortscale'}),
    'scale_down': (1, 1, {'constant'}),
    'vector_neut': (2, 2, {}),
    'winsorize': (1, 1, {'std'}),
    # --- vector ---
    'vec_avg': (1, 1, {}), 'vec_count': (1, 1, {}), 'vec_max': (1, 1, {}),
    'vec_min': (1, 1, {}), 'vec_range': (1, 1, {}),
    'vec_stddev': (1, 1, {}), 'vec_sum': (1, 1, {}),
    # --- transformational / group ---
    'bucket': (1, 1, {'range', 'buckets', 'skipBoth', 'NaNGroup'}),
    'trade_when': (3, 3, {}),
    'group_backfill': (3, 3, {'std'}),
    'group_cartesian_product': (2, 2, {}),
    'group_mean': (3, 3, {}), 'group_neutralize': (2, 2, {}),
    'group_scale': (2, 2, {}), 'group_zscore': (2, 2, {}),
    'group_max': (2, 2, {}), 'group_min': (2, 2, {}),
    'group_rank': (2, 2, {}),
}

LOOKBACK_OPS = ({o for o in OPS_RULES if o.startswith('ts_')}
                | {'last_diff_value', 'kth_element'})

# ops whose second arg is NOT a window (fill value / step size etc.)
TS_SKIP = {'ts_backfill', 'ts_step', 'ts_fill_nan'}

# default lookback when a ts_* call is missing its window arg
TS_DEFAULT_LOOKBACK = {op: 20 for op in LOOKBACK_OPS if op not in TS_SKIP}

_SKIP_TOKENS = (set(fields) | set(OPS_RULES) |
    {'subindustry', 'industry', 'sector', 'market', 'exchange', 'country',
     'cap', 'volume', 'adv20', 'open', 'close', 'high', 'low', 'vwap', 'returns',
     'range', 'filter', 'dense', 'factor', 'percentage', 'driver', 'mode',
     'k', 'lag', 'rettype', 'hump', 'p', 'sensitivity', 'force', 'rate', 'c',
     'x', 'y', 'z', 'alpha', 'beta', 'gamma', 'theta', 'lambda'})

# ----------------------------- helpers --------------------------------------

def _normalize(expr):
    """Whitespace-tolerant normalization: (a,100) == (a, 100).
    Validation AND repair both operate on the normalized form, so what is
    validated is exactly what gets simulated."""
    out = ' '.join(expr.split())
    out = re.sub(r'\(\s+', '(', out)
    out = re.sub(r'\s+\)', ')', out)
    out = re.sub(r'\s*,\s*', ', ', out)
    return out.strip()


def _parse_json_array(text):
    """Robustly parse a JSON array from model output:
    clean -> sliced -> single-quoted dialect -> regex-extracted embedded array."""
    t = text.strip()
    try:
        return json.loads(t)
    except Exception:
        pass
    s, e = t.find('['), t.rfind(']')
    if s != -1 and e > s:
        cand = t[s:e + 1]
        for parse in (json.loads, ast.literal_eval):
            try:
                return parse(cand)
            except Exception:
                pass
    # fallback: find an embedded [{...},{...}] anywhere in prose/reasoning
    m = re.search(r'\[\s*\{.*\}\s*\]', t, re.DOTALL)
    if m:
        cand = m.group(0)
        for parse in (json.loads, ast.literal_eval):
            try:
                return parse(cand)
            except Exception:
                pass
    raise ValueError(f'Could not parse JSON array from model output.\n'
                     f'First 500 chars: {t[:500]!r}')


def repair_entries(expr):
    """Expand LLM shorthand entry_<fieldid> to the real wrapper."""
    out = expr
    for f in fields:
        out = re.sub(rf'(?i)\bentry_{re.escape(f)}\b', field_base[f], out)
    return out


def _calls(expr):
    for m in re.finditer(r'([A-Za-z_][A-Za-z0-9_]*)\s*\(', expr):
        depth, i = 1, m.end()
        while i < len(expr) and depth:
            depth += (expr[i] == '(') - (expr[i] == ')')
            i += 1
        yield m.group(1), expr[m.end():i - 1]


def _top_args(args_str):
    out, depth, cur = [], 0, ''
    for ch in args_str:
        depth += (ch == '(') - (ch == ')')
        if ch == ',' and depth == 0:
            out.append(cur.strip()); cur = ''
        else:
            cur += ch
    out.append(cur.strip())
    return [a for a in out if a]


def _int_literals(args_str):
    return [a for a in _top_args(args_str) if a.isdigit()]


def _op_problems(op, args):
    """Doc-verified arity + keyword-only enforcement."""
    p = []
    pos = [a for a in args if '=' not in a]
    kws = [a.split('=')[0].strip() for a in args if '=' in a]
    if op not in OPS_RULES:
        p.append(f'operator "{op}" is not in the allowed list')
        return p
    pmin, pmax, kw_ok = OPS_RULES[op]
    if len(pos) < pmin:
        p.append(f'{op}(...) has {len(pos)} input(s), needs >= {pmin}')
    if pmax is not None and len(pos) > pmax:
        p.append(f'{op}(...) has {len(pos)} positional input(s), needs <= {pmax} '
                 f'(options "{sorted(kw_ok) or chr(39)+chr(39)}" must be keyword form)')
    for k in kws:
        if k not in kw_ok:
            p.append(f'{op}(...): unknown keyword "{k}" - valid: {sorted(kw_ok) or "none"}')
    # positional floats only in true arithmetic
    if op not in TRUE_ARITH:
        for a in pos:
            if ('.' in a) and a.replace('.', '', 1).isdigit():
                p.append(f'{op}(...): positional float "{a}" not valid here (use keyword form)')
    return p


# ------------------------- paren-aware scan utilities -------------------------

def _find_call_end(s, open_idx):
    """Given index of '(', return index of its matching ')'."""
    depth = 0
    for i in range(open_idx, len(s)):
        if s[i] == '(':
            depth += 1
        elif s[i] == ')':
            depth -= 1
            if depth == 0:
                return i
    return -1


def _split_top_args(argstr):
    """Split argument string on top-level commas (paren-aware)."""
    args, depth, cur = [], 0, ''
    for ch in argstr:
        if ch == '(':
            depth += 1
        elif ch == ')':
            depth -= 1
        if ch == ',' and depth == 0:
            args.append(cur.strip())
            cur = ''
        else:
            cur += ch
    if cur.strip():
        args.append(cur.strip())
    return args


# ----------------------------- AUTO-REPAIR (fixpoint) -------------------------

def _insert_missing_lookbacks(expr):
    """One pass: for every ts_*(x) call missing its window arg, insert the default.
    Paren-aware: handles nesting and never touches keyword args."""
    out, i, changed = [], 0, False
    while i < len(expr):
        m = re.match(r'(ts_[a-z_]+)\s*\(', expr[i:])
        if m and m.group(1) in TS_DEFAULT_LOOKBACK:
            name = m.group(1)
            open_idx = i + m.end() - 1
            close_idx = _find_call_end(expr, open_idx)
            if close_idx != -1:
                argstr = expr[open_idx + 1:close_idx]
                args = _split_top_args(argstr)
                if len(args) == 1:
                    out.append(f'{name}({args[0]}, {TS_DEFAULT_LOOKBACK[name]})')
                    changed = True
                    i = close_idx + 1
                    continue
        out.append(expr[i])
        i += 1
    return ''.join(out), changed


def _wrap_bare_fields(expr):
    """One pass: bare field name -> ts_backfill(field, BACKFILL_D),
    except occurrences already inside a ts_backfill(...) call."""
    _WRAP_SPLIT = re.compile(r'(ts_backfill\([^()]*\))')     # top-level wrapper chunks
    parts = _WRAP_SPLIT.split(expr)
    out = ''
    changed = False
    for seg in parts:
        if seg.startswith('ts_backfill('):
            out += seg
            continue
        def _sub(m):
            nonlocal changed
            changed = True
            return f'ts_backfill({m.group(0)}, {BACKFILL_D})'
        for f in fields:
            new_seg = re.sub(rf'(?<![A-Za-z0-9_]){re.escape(f)}(?![A-Za-z0-9_])', _sub, seg)
            if new_seg != seg:
                seg = new_seg
        out += seg
    return out, changed


def _one_repair_pass(expr):
    """A single repair transform. Order matters: wrap fields first (that can
    create a one-arg ts_* call), then fix lookbacks."""
    nxt, changed = _wrap_bare_fields(expr)
    if changed:
        return nxt, True
    return _insert_missing_lookbacks(expr)


def auto_repair(expr, max_iter=6):
    """Repair until validate() passes or no progress can be made.
    Each pass is re-validated, so multi-step chains like
        rank(ts_zscore(FIELD)) -> rank(ts_zscore(ts_backfill(FIELD, 100), 20))
    complete in sequence instead of the first repair short-circuiting."""
    cur = _normalize(expr)
    for _ in range(max_iter):
        if not validate(cur):
            return cur
        nxt, changed = _one_repair_pass(cur)
        if not changed or nxt == cur:
            break                      # stuck - return best effort
        cur = nxt
    return cur


# ----------------------------- VALIDATE --------------------------------------

def validate(expr):
    p = []
    if not expr.strip():
        return ['empty expression']
    if expr.count('(') != expr.count(')'):
        p.append('unbalanced parentheses')

    # normalize so spacing variants match the canonical wrappers
    core = _normalize(expr)
    for f in fields:
        core = core.replace(field_base[f], 'PLACEHOLDER')
    if 'PLACEHOLDER' not in core:
        p.append('no datafield referenced - copy the full wrapper expression verbatim')

    kw_names = set()                       # kw names are never variables
    for op, argstr in _calls(core):
        args = _top_args(argstr)
        for a in args:
            if '=' in a:
                kw_names.add(a.split('=')[0].strip())

        # tuple args — FastExpr has NO tuples
        for a in args:
            if a.startswith('(') and ',' in a and a.endswith(')'):
                p.append(f'{op}(...): parenthesized multi-value argument is a tuple - invalid')

        # doc-verified arity + keywords
        p += _op_problems(op, args)

        # integer lookback presence for ts_* / d-based ops
        if op in LOOKBACK_OPS and not _int_literals(argstr):
            p.append(f'{op}(...) needs an integer lookback, e.g. {op}(..., 20)')

    bare = re.sub(r'"[^"]*"', '', core)
    used_ops = {op for op, _ in _calls(core)}
    for tok in set(re.findall(r'[A-Za-z_][A-Za-z0-9_]*', bare)):
        if tok == 'PLACEHOLDER' or tok in used_ops or tok in _SKIP_TOKENS \
           or tok in kw_names:
            continue
        p.append(f'unknown variable "{tok}" - copy the full wrapper expression, never invent names')
    for f in fields:
        if re.search(rf'(?<![A-Za-z0-9_]){re.escape(f)}(?![A-Za-z0-9_])', bare):
            p.append(f'bare field "{f}" used - it may appear ONLY inside its full wrapper expression')
    return p


# ----------------------------- GENERATE-VALID --------------------------------

def generate_valid(history, n=10, max_tries=2):
    # ---- phase 1: generation, tolerant of empty/garbage replies ----
    proposals = []
    for attempt in range(1, max_tries + 1):
        try:
            first = generate(history, n)
        except Exception as e:                       # parse errors, timeouts, etc.
            print(f'[generate] attempt {attempt}/{max_tries} FAILED: {e!r}', flush=True)
            first = []
        proposals = list(first)
        if proposals:
            break
        print(f'[generate] attempt {attempt}/{max_tries}: no usable proposals', flush=True)
    if not proposals:
        print('[generate] GAVE UP - returning empty batch (nothing spent)', flush=True)
        return []

    for pr in proposals:                             # repair BEFORE validating
        pr['expression'] = auto_repair(repair_entries(pr['expression']))

    # ---- phase 2: validate + refill (refill failures are non-fatal) ----
    for _ in range(max_tries):
        bad  = [pr for pr in proposals if validate(pr['expression'])]
        good = [pr for pr in proposals if not validate(pr['expression'])]
        if not bad:
            break
        for pr in bad:
            print(f'[VALIDATOR] rejected: {pr["expression"][:90]}')
            print(f'            reason   : {validate(pr["expression"])}')
        try:
            refill = llm.invoke([
                SystemMessage(content='Return only a raw JSON array of objects with keys "expression" and "rationale". No commentary.'),
                HumanMessage(content=(
                    f'These proposals were REJECTED for syntax: '
                    f'{[(pr["expression"], validate(pr["expression"])) for pr in bad]}. '
                    f'Fix each one respecting all previous rules. Every field reference '
                    f'MUST be the full wrapper ts_backfill(<field>, {BACKFILL_D}) - '
                    f'never the bare field name. Return ONLY a JSON array of '
                    f'{len(bad)} corrected objects.')),
            ])
            text = _content_to_text(refill.content).strip().replace('```json', '').replace('```', '').strip()
            raw = _parse_json_array(text)
        except Exception as e:
            print(f'[VALIDATOR] refill FAILED ({e!r}) -> dropping {len(bad)} bad slot(s)', flush=True)
            raw = []
        fixed = [{'expression': str(i.get('expression', '')).strip(),
                  'rationale': str(i.get('rationale', '')).strip()}
                 for i in raw if isinstance(i, dict)]
        for pr in fixed:                             # repair refills too
            pr['expression'] = auto_repair(repair_entries(pr['expression']))
        proposals = good + fixed

    final = [pr for pr in proposals if not validate(pr['expression'])]
    if not final:
        print('[VALIDATOR] WARNING: no proposals survived validation', flush=True)
    return final


# ----------------------------- SELF-TESTS ------------------------------------
def _selftest():
    f0 = fields[0]
    w  = field_base[f0]

    # canonical forms pass
    assert validate(f'rank(ts_zscore({w}, 20))') == [], \
        f'canonical wrapper must pass: {validate(f"rank(ts_zscore({w}, 20))")}'

    # whitespace variants pass (normalization)
    assert validate(f'ts_zscore({w},20)') == [], 'no-space lookback must pass (normalized)'

    # bare field still caught when NOT repaired
    errs = validate(f'ts_zscore({f0}, 20)')
    assert any('bare field' in x for x in errs), f'bare field must be caught: {errs}'

    # STEP 1: wrapping alone must be correct on a pre-valid expression
    r1, changed = _wrap_bare_fields(f'ts_zscore({f0}, 20)')
    assert changed and validate(r1) == [], f'wrap step broken: {r1} | {validate(r1)}'

    # STEP 2: lookback insert alone must be correct on a pre-wrapped expression
    r2, changed = _insert_missing_lookbacks(f'ts_zscore({w})')
    assert changed and validate(r2) == [], f'lookback step broken: {r2} | {validate(r2)}'

    # FULL CHAIN: bare field AND missing lookback together
    r3 = auto_repair(f'rank(ts_zscore({f0}))')
    assert validate(r3) == [], f'auto_repair fixpoint broken: {r3} | {validate(r3)}'
    assert f'ts_backfill({f0}, {BACKFILL_D})' in r3 and re.search(r',\s*20\)', r3), \
        f'auto_repair must chain both steps: {r3}'

    # ts_backfill must NEVER get a spurious default lookback appended
    r4 = _insert_missing_lookbacks(w)[0]
    assert r4 == w, f'ts_backfill must not be touched: {r4}'

    # keyword names must never be flagged as unknown variables
    kw_test = f'ts_scale({w}, constant=1)'
    assert not any('unknown' in x and 'constant' in x for x in validate(kw_test)), \
        f'kw name "constant" wrongly flagged: {validate(kw_test)}'

    print('validator v7 self-tests OK')

_selftest()

validator v7 self-tests OK


In [47]:
def simulate_and_score(proposals):
    """Build payloads (expression in regular=...), apply NEU as a setting, RUN them.
    Uses SIMULATION_MODE ('QUICK' or 'FULL') for every payload.
    Tags every result with its full context (region/universe/delay/decay/neu/dataset).
    QUICK-mode guard: alphas whose stats come back None/non-numeric are SKIPPED
    loudly (not added to history) instead of crashing the loop."""
    if proposals and isinstance(proposals[0], str):          # backwards compat
        proposals = [{'expression': e, 'rationale': ''} for e in proposals]

    ctx = f"{REGION}|{UNIVERSE}|D{DELAY}|dec{DECAY}|{NEU}|{DATASET}"

    payloads = [ace.generate_alpha(regular=p['expression'], region=REGION, universe=UNIVERSE,
                                   delay=DELAY, decay=DECAY,
                                   neutralization=NEU, max_trade=MAX_TRADE,
                                   simulation_mode=SIMULATION_MODE)
                for p in proposals]
    rationale_of = {p['expression']: p['rationale'] for p in proposals}   # order-safe lookup

    out = []
    for item in ace.simulate_alpha_list_multi(s, payloads):
        expr = item['simulate_data']['regular']
        if item['alpha_id'] is None:
            out.append({'expression': expr, 'alpha_id': None,
                        'sharpe': -9.0, 'fitness': -9.0, 'turnover': 0.0,
                        'rationale': rationale_of.get(expr, ''), 'ctx': ctx})
            continue
        row = item['is_stats'].iloc[0]
        # --- QUICK-mode guard: BRAIN may return None for some stats ----
        try:
            _s = float(row['sharpe'])
            _f = float(row['fitness'])
            _t = float(row['turnover'])
        except (TypeError, ValueError, KeyError) as _e:
            print(f'[sim] alpha {item["alpha_id"]} returned non-numeric stats '
                  f'({_e!r}) -> marking as FAILED, removing from history', flush=True)
            continue
        out.append({'expression': expr, 'alpha_id': item['alpha_id'],
                    'sharpe': _s, 'fitness': _f, 'turnover': _t,
                    'rationale': rationale_of.get(expr, ''), 'ctx': ctx})
    return out

RUN CELL

In [48]:
# RUN - loop with crowding measurement + isolated retry of failed sims.
# Re-run this cell (then VIEW) to continue. History ledgers live in ./history/.
# Retries are attempted FIRST each iteration, one at a time (poison-proof),
# then fresh proposals run as a normal multi-sim batch.
import os
import shutil
from pathlib import Path
import pandas as pd

# --- PER-CONTEXT history: one ledger per dataset+region+delay ----------------
HISTORY_DIR = Path('history')
HISTORY_DIR.mkdir(exist_ok=True)                      # fail loud on permission errors

HISTORY_FILE = HISTORY_DIR / f"history_{REGION}_{UNIVERSE}_D{DELAY}_{DATASET}.csv"

# one-time migration of ALL legacy ledgers (idempotent: only moves if target missing)
for _legacy in Path('.').glob('history_*.csv'):
    _target = HISTORY_DIR / _legacy.name
    if not _target.exists():
        shutil.move(str(_legacy), str(_target))
        print(f'[history] migrated {_legacy} -> {_target}')

# --- load prior progress (survives kernel restarts / crashes) ---------------
history = [] if 'history' not in globals() else history
ctx = f"{REGION}|{UNIVERSE}|D{DELAY}|dec{DECAY}|{NEU}|{DATASET}"

history = [r for r in history
           if r.get('ctx') == ctx
           or any(f in str(r.get('expression', '')) for f in (fields or []))]

if os.path.exists(HISTORY_FILE) and not history:      # load from disk only if empty
    try:
        history = pd.read_csv(HISTORY_FILE).to_dict('records')
        for r in history:
            aid = r.get('alpha_id')
            r['alpha_id'] = (None if aid is None or (isinstance(aid, float) and pd.isna(aid)) else aid)
        print(f'Loaded {len(history)} prior results from {HISTORY_FILE}')
    except Exception as e:
        print(f'WARNING: could not load {HISTORY_FILE} ({e!r}) - starting fresh')

N_ITERATIONS = 5

def _save_history():
    """Single save point. write-to-temp + atomic rename so a crash mid-write
    can never corrupt the ledger (fail-loud invariant: state must survive)."""
    tmp = HISTORY_FILE.with_suffix('.tmp')
    pd.DataFrame(history).to_csv(tmp, index=False)
    tmp.replace(HISTORY_FILE)

for iteration in range(1, N_ITERATIONS + 1):
    # --- session countdown: know BEFORE you spend a batch ---
    try:
        _rem = ace.check_session_timeout(s)
        print(f'[session] remaining ~{_rem/60:.0f} min', flush=True)
        if _rem < 2500:
            print('WARNING: session expires soon - finish this iteration, then re-run '
                  'Setup before continuing.', flush=True)
    except Exception:
        pass

    try:
        print(f'\n===== Iteration {iteration} | dataset {DATASET} | {len(fields)} fields '
              f'| retry queue: {retry_queue_size()} =====', flush=True)

        # ---- 1. ISOLATED RETRY BATCH (failed sims from previous iterations) ----
        if retry_queue_size():
            print('\n--- retrying previously-failed alpha(s) ---', flush=True)
            rres = run_retries_isolated(RETRY_PER_ITER)
            if rres:
                history.extend(rres)
                _save_history()                 # SAVE FIRST
                enqueue_failed(rres)            # success -> drop, failure -> attempt++
                for r in rres:
                    tag = 'OK' if r.get('alpha_id') else 'STILL FAILING'
                    print(f'  [retry] {tag}: {r["expression"][:70]}', flush=True)

        # ---- 2. FRESH PROPOSALS ----
        proposals = generate_valid(history)
        if not proposals:
            print('  no valid proposals -> skipping fresh batch (nothing spent)', flush=True)
            continue
        print(f'\n--- {len(proposals)} proposed alphas ---')
        for i, p in enumerate(proposals, 1):
            print(f'\n[{i:2d}] {p["expression"]}')
            print(f'     rationale : {p["rationale"]}')

        results = simulate_and_score(proposals)

        history.extend(results)                                  # SAVE FIRST
        _save_history()

        n_ok = sum(1 for r in results if r.get('alpha_id'))
        if n_ok == 0:
            print(f'  WARNING: all {len(results)} proposal(s) FAILED -> queued for retry, '
                  f'nothing to rank.', flush=True)
        enqueue_failed(results)                                  # queue fresh failures

        measure_topk(results)                                    # then measure (non-fatal)

        meas = [r for r in results if r.get('prod_corr') is not None]
        if meas:
            print('\n--- crowding check (top-K of this batch) ---')
            for r in meas:
                flag = ('CROWDED -> agent must change the idea, not the syntax'
                        if r['prod_corr'] >= PROD_CORR_LIMIT else 'ok')
                print(f"  prod-corr {r['prod_corr']:.2f} [{flag}] <- "
                      f"{r['alpha_id']}  {r['expression'][:65]}")

        best = best_uncrowded(history)
        if best is None:
            print('\nbest so far: (none - no successful alpha in this context yet)', flush=True)
        else:
            if best.get('prod_corr') is not None and best['prod_corr'] >= PROD_CORR_LIMIT:
                print('  NOTE: all measured alphas are CROWDED -> showing overall best as fallback',
                      flush=True)
            bpc = best.get('prod_corr')
            pc_txt = (f" | prod-corr {bpc:.2f}"
                      + (' [CROWDED]' if bpc is not None and bpc >= PROD_CORR_LIMIT else '')
                      if bpc is not None else '')
            print(f'\nbest so far: {best["alpha_id"]} | sharpe {best["sharpe"]:.2f} | '
                  f'fitness {best["fitness"]:.2f}{pc_txt} <- {best["expression"][:60]}')

    except SessionExpired as e:
        _save_history()                                          # state is safe
        print('\n' + str(e))
        print(f'SAVED {len(history)} alphas to {HISTORY_FILE} - nothing lost.')
        break

print('\nDONE -', len(history), 'alphas simulated.',
      '| retry queue remaining:', retry_queue_size())

Loaded 156 prior results from history/history_ASI_TOP500_D1_fundamental17.csv
[session] remaining ~203 min
[retry] loaded 0 queued failure(s) from history/retry_ASI_TOP500_D1_fundamental17.json

===== Iteration 1 | dataset fundamental17 | 20 fields | retry queue: 0 =====
[generate] attempt 1/2 FAILED: ValueError("Could not parse JSON array from model output.\nFirst 500 chars: 'Let me analyze the history and strategy carefully.\\n\\nBest fitness so far: 0.58 (expression #1)\\nThe best-performing signal family involves FX correlation deltas, particularly around `fnd17_5_usdtorepexrate` and `fnd17_8_reptoprcexrate` / `fnd17_7_reptoprcexrate` pairs.\\n\\nKey successful patterns:\\n- ts_delta(ts_corr(ts_backfill(...), ts_backfill(...), N), M) - correlation delta\\n- ts_zscore applied to various transforms\\n- subtract for mean-reversion residuals\\n- scale/divide for normalization\\n\\nStrategy'")
[generate] attempt 1/2: no usable proposals
[VALIDATOR] rejected: ts_delta(ts_corr(ts_backfill

KeyboardInterrupt: 

In [32]:
# VISUALIZE - PnL of the best alpha (highest fitness)
import plotly.express as px

qualified = [r for r in history if r.get('alpha_id')]     # only successfully simulated alphas
if not qualified:
    raise ValueError('No successfully simulated alphas yet - run the loop first.')

best = max(qualified, key=lambda r: r['fitness'])
alpha_id = best['alpha_id']
print(f"Best alpha : {alpha_id} | sharpe {best['sharpe']:.2f} | fitness {best['fitness']:.2f}")
print(f"expression : {best['expression']}")

alpha_pnl = ace.get_alpha_pnl(s, alpha_id)
if alpha_pnl.empty:
    raise ValueError(f'No PnL records returned for {alpha_id}')

pnl_cols = ['pnl', 'risk-neutralized-pnl', 'investability-constrained-pnl']
pnl_cols = [c for c in pnl_cols if c in alpha_pnl.columns]      # guard against API changes

fig = px.line(
    alpha_pnl, x=alpha_pnl.index, y=pnl_cols,
    labels={'x': 'Date', 'value': 'PnL ($)', 'variable': 'series'},
)
fig.add_hline(y=0, line_dash='dot', line_color='gray')
fig.update_layout(
    title=dict(
        text=(f"<b>alpha_id={hf.make_clickable_alpha_id(alpha_id)}</b><br>"
              f"<sup>sharpe {best['sharpe']:.2f} | fitness {best['fitness']:.2f} | "
              f"turnover {best['turnover']:.2%}</sup><br>"
              f"{best['expression']}"),
        x=0.5,
    ),
    xaxis_title='Date',
    yaxis_title='PnL ($)',
    legend_title_text='',
    hovermode='x unified',
)
fig.show()

Best alpha : xA3nnalJ | sharpe 3.59 | fitness 2.86
expression : rank(ts_zscore(ts_corr(ts_backfill(vec_avg(news_geographic_area_livenews_all), 100), ts_backfill(vec_avg(news_product_code_all), 100), 60), 20))


In [33]:
# OPTIONAL - top-3 alphas, raw PnL side by side
import functools

top3 = sorted(qualified, key=lambda r: r['fitness'], reverse=True)[:5]
frames = []
for r in top3:
    p = ace.get_alpha_pnl(s, r['alpha_id'])
    if not p.empty:
        frames.append(p[['pnl']].rename(columns={'pnl': r['alpha_id']}))
        print(f"{r['alpha_id']}  fitness {r['fitness']:.2f}  <- {r['expression'][:70]}")

merged = functools.reduce(lambda a, b: a.join(b, how='outer'), frames).sort_index()
fig = px.line(merged, x=merged.index, y=list(merged.columns),
              labels={'x': 'Date', 'value': 'PnL ($)'})
fig.add_hline(y=0, line_dash='dot', line_color='gray')
fig.update_layout(title='<b>Top-3 alphas by fitness — cumulative PnL</b>',
                  xaxis_title='Date', yaxis_title='PnL ($)', hovermode='x unified')
fig.show()

xA3nnalJ  fitness 2.86  <- rank(ts_zscore(ts_corr(ts_backfill(vec_avg(news_geographic_area_livene
A1N36Prw  fitness 0.41  <- rank(ts_zscore(days_from_last_change(ts_backfill(rsk68_weight_edadv, 1
9qjR1Q79  fitness 0.29  <- multiply(ts_backfill(rsk66_countrysector_basedata_cap_usd, 100), ts_ba
e7bLM8m6  fitness 0.26  <- ts_backfill(rsk66_countrysector_basedata_cap_usd, 100)
blbv02Xm  fitness 0.26  <- ts_backfill(rsk66_globalindustry_basedata_cap_usd, 100)


In [ ]:
# # VIEW - leaderboard with crowding status + links
# board = pd.DataFrame(history).sort_values('fitness', ascending=False).reset_index(drop=True)
# if 'prod_corr' in board.columns:
#     board['crowd_status'] = board['prod_corr'].map(
#         lambda v: 'CROWDED' if pd.notna(v) and v >= PROD_CORR_LIMIT else 'ok')
# pd.set_option('max_colwidth', None)
# cols = ['fitness', 'sharpe', 'turnover', 'prod_corr', 'crowd_status', 'alpha_id', 'expression']
# display(board[[c for c in cols if c in board.columns]])

# for r in history:
#     if r['alpha_id']:
#         print(f"https://platform.worldquantbrain.com/alpha/{r['alpha_id']}  <- {r['expression']}")

In [145]:
# ============================================================================
# ALPHA REPORT — yearly metrics + correlations + PnL visualization
# Usage:  report = alpha_report(s, alpha_id)      # displays everything, returns data
# ============================================================================
import pandas as pd
import plotly.express as px
import ace_lib as ace
import helpful_functions as hf


def alpha_report(s, alpha_id, self_corr_threshold=0.7, prod_corr_threshold=0.7):
    """Detailed performance report for one alpha.

    Displays:
      1. Header (settings, expression, platform link)
      2. Year-over-year table: sharpe, fitness, turnover, returns, drawdown, margin
         (+ an 'ALL (IS)' aggregate row)
      3. Correlation summary: self-corr & prod-corr vs thresholds
      4. PnL chart: pnl / risk-neutralized-pnl / investability-constrained-pnl

    Returns a dict with all underlying dataframes for further use.
    """
    # ---------- 1. aggregate IS stats + settings + expression ----------
    result   = ace.get_simulation_result_json(s, alpha_id)
    iss      = result.get('is', {}) or {}
    settings = result.get('settings', {}) or {}
    expr     = (result.get('regular') or {}).get('code', '')

    print('=' * 100)
    print(f"ALPHA REPORT  |  id: {alpha_id}")
    print('=' * 100)
    print(f"link        : https://platform.worldquantbrain.com/alpha/{alpha_id}")
    print(f"settings    : {settings.get('region')}/{settings.get('universe')} | "
          f"delay {settings.get('delay')} | decay {settings.get('decay')} | "
          f"neutralization {settings.get('neutralization')}")
    print(f"expression  : {expr}")

    # ---------- 2. year-over-year table ----------
    ydf = ace.get_alpha_yearly_stats(s, alpha_id)
    metric_cols = ['sharpe', 'fitness', 'turnover', 'returns', 'drawdown', 'margin']
    yearly = ydf[[c for c in ['year', 'stage'] + metric_cols if c in ydf.columns]].copy()

    overall = {c: iss.get(c) for c in metric_cols}
    overall.update({'year': 'ALL (IS)', 'stage': ''})
    table = pd.concat([pd.DataFrame([overall]), yearly], ignore_index=True)

    disp = table.copy()
    for c in ['sharpe', 'fitness', 'turnover']:
        if c in disp:
            disp[c] = pd.to_numeric(disp[c], errors='coerce').round(2)
    for c in ['returns', 'drawdown']:
        if c in disp:
            disp[c] = (pd.to_numeric(disp[c], errors='coerce') * 100).round(2).astype(str) + '%'
    if 'margin' in disp:
        disp['margin'] = pd.to_numeric(disp['margin'], errors='coerce').round(6)

    print(f"\n--- performance by year ({len(yearly)} rows + aggregate) ---")
    display(disp)

    # ---------- 3. correlations ----------
    scdf = ace.get_self_corr(s, alpha_id)
    pcdf = ace.get_prod_corr(s, alpha_id)

    self_val = (float(scdf['correlation'].max()) if not scdf.empty and 'correlation' in scdf
                else 0.0)
    prod_val = 0.0
    if not pcdf.empty and 'alphas' in pcdf.columns and (pcdf['alphas'] > 0).any():
        prod_val = float(pcdf[pcdf['alphas'] > 0]['max'].max())

    summary = pd.DataFrame([
        {'metric': 'sharpe (IS)',   'value': round(iss.get('sharpe') or 0, 2)},
        {'metric': 'fitness (IS)',  'value': round(iss.get('fitness') or 0, 2)},
        {'metric': 'turnover (IS)', 'value': f"{(iss.get('turnover') or 0):.2%}"},
        {'metric': 'returns (IS)',  'value': f"{(iss.get('returns') or 0):.2%}"},
        {'metric': 'drawdown (IS)', 'value': f"{(iss.get('drawdown') or 0):.2%}"},
        {'metric': 'margin (IS)',   'value': f"{(iss.get('margin') or 0):.6f}"},
        {'metric': 'self correlation', 'value': round(self_val, 4),
         'limit': self_corr_threshold,
         'status': 'PASS' if self_val <  self_corr_threshold else 'FAIL'},
        {'metric': 'prod correlation', 'value': round(prod_val, 4),
         'limit': prod_corr_threshold,
         'status': 'PASS' if prod_val <= prod_corr_threshold else 'FAIL'},
    ])
    print("\n--- summary & submission gates ---")
    display(summary)

    if not scdf.empty:                       # worst self-corr offenders, if any
        cols = [c for c in ['id', 'region', 'correlation', 'sharpe', 'fitness']
                if c in scdf.columns]
        top = scdf.sort_values('correlation', ascending=False).head(5)[cols]
        print("top-5 self-correlated alphas:")
        display(top)
    else:
        print("self correlation: no correlated alphas found (effectively 0).")

    # ---------- 4. PnL visualization ----------
    pnl = ace.get_alpha_pnl(s, alpha_id)
    if not pnl.empty:
        pnl_cols = [c for c in ['pnl', 'risk-neutralized-pnl',
                                'investability-constrained-pnl'] if c in pnl.columns]
        fig = px.line(pnl, x=pnl.index, y=pnl_cols,
                      labels={'x': 'Date', 'value': 'PnL ($)', 'variable': 'series'})
        fig.add_hline(y=0, line_dash='dot', line_color='gray')
        fig.update_layout(
            title=dict(
                text=(f"<b>alpha_id={hf.make_clickable_alpha_id(alpha_id)}</b><br>"
                      f"<sup>sharpe {iss.get('sharpe')} | fitness {iss.get('fitness')} | "
                      f"turnover {iss.get('turnover')}</sup><br>{expr}"),
                x=0.5),
            xaxis_title='Date', yaxis_title='PnL ($)',
            legend_title_text='', hovermode='x unified')
        fig.show()
    else:
        print("no PnL records returned.")

    print('=' * 100)
    return {'summary': summary, 'yearly': ydf, 'overall': overall,
            'self_corr': scdf, 'prod_corr': pcdf, 'pnl': pnl, 'result': result}

In [10]:
rp = alpha_report(s,"KPNxROZE")

ALPHA REPORT  |  id: KPNxROZE
link        : https://platform.worldquantbrain.com/alpha/KPNxROZE
settings    : USA/TOP2000 | delay 1 | decay 4 | neutralization REVERSION_AND_MOMENTUM
expression  : divide(ts_backfill(vec_avg(section_positive_score_2), 100), add(ts_backfill(vec_avg(section_negative_score_2), 100), ts_backfill(vec_avg(key_factor_negative_score_section_2), 100)))

--- performance by year (10 rows + aggregate) ---


,sharpe,fitness,turnover,returns,drawdown,margin,year,stage
0,1.08,0.67,0.09,4.78%,7.15%,0.001100,ALL (IS),
1,0.44,0.16,0.09,1.56%,5.23%,0.000361,2014,IS
2,0.92,0.47,0.08,3.31%,4.25%,0.000858,2015,IS
3,0.91,0.52,0.09,4.02%,3.21%,0.000912,2016,IS
4,2.53,2.10,0.08,8.59%,2.31%,0.002190,2017,IS
5,1.25,0.73,0.08,4.21%,3.31%,0.001044,2018,IS
6,-1.07,-0.59,0.09,-3.85%,5.27%,-0.000903,2019,IS
7,2.74,3.25,0.10,17.61%,4.44%,0.003598,2020,IS
8,1.82,1.48,0.09,8.28%,2.41%,0.001846,2021,IS
9,1.09,0.74,0.09,5.77%,2.94%,0.001283,2022,IS



--- summary & submission gates ---


,metric,value,limit,status
0,sharpe (IS),1.08,NaN,NaN
1,fitness (IS),0.67,NaN,NaN
2,turnover (IS),8.70%,NaN,NaN
3,returns (IS),4.78%,NaN,NaN
4,drawdown (IS),7.15%,NaN,NaN
5,margin (IS),0.001100,NaN,NaN
6,self correlation,0.2621,0.7,PASS
7,prod correlation,0.6,0.7,PASS


top-5 self-correlated alphas:


,id,region,correlation,sharpe,fitness
0,LRVg6vM,USA,0.2621,1.28,1.24
1,LRjxKp1,USA,0.2540,1.30,1.00
2,A3WKzYw,USA,0.2504,1.94,1.80
3,xRWoRaN,USA,0.2208,1.59,1.14
4,GLdMoqP,USA,0.2032,2.21,1.43


kqo2A0NL
npdMxYpq

In [ ]:
# ============================================================================
# SIMULATE & REPORT ONE ALPHA — from a raw expression to a full report
# Usage:
#   report = simulate_and_report(
#       s,
#       expression="rank(ts_zscore(ts_backfill(vec_avg(section_positive_score), 100), 20))",
#       region="USA", universe="TOP2000", delay=1, decay=4,
#       neutralization="REVERSION_AND_MOMENTUM", max_trade="OFF",
#   )
# ============================================================================
import pandas as pd
import ace_lib as ace


def simulate_and_report(s, expression, region="USA", universe="TOP3000",
                        delay=1, decay=4, neutralization="SUBINDUSTRY",
                        max_trade="OFF", **report_kwargs):
    """Simulate ONE alpha from an expression, then display a full performance report.

    Args:
        s              : authenticated BRAIN session (ace.start_session())
        expression     : the alpha expression string
        region/universe/delay/decay/neutralization/max_trade : simulation settings
        **report_kwargs: forwarded to alpha_report (e.g. prod_corr_threshold=0.7)

    Returns the alpha_report dict (summary, yearly, self_corr, prod_corr, pnl, result).
    """
    # 1) build the payload — expression goes in `regular`, settings in the payload
    payload = ace.generate_alpha(
        regular=expression,
        region=region,
        universe=universe,
        delay=delay,
        decay=decay,
        neutralization=neutralization,
        max_trade=max_trade,
        simualtion_mode="FULL",
    )
    print(f"[simulate] launching: {expression}")
    print(f"[simulate] {region}/{universe} | delay {delay} | decay {decay} | "
          f"neu {neutralization} | max_trade {max_trade}")

    # 2) run it (single-alpha path of the same machinery the loop uses)
    outcome = ace.simulate_single_alpha(s, payload)

    if outcome.get("alpha_id") is None:
        print("=" * 100)
        print("SIMULATION FAILED — invalid syntax / unknown field / bad operator.")
        print("Check the ace log lines just above for the exact error message.")
        print("=" * 100)
        return None

    alpha_id = outcome["alpha_id"]
    print(f"[simulate] OK -> alpha_id {alpha_id}")

    # 3) reuse the full report
    return alpha_report(s, alpha_id, **report_kwargs)

In [150]:


# try one expression by hand:
r = simulate_and_report(
    s,
    region="USA", universe="TOP3000", delay=1, decay=2,neutralization="INDUSTRY", max_trade="OFF",
    expression="""

    rank(ts_zscore(divide(ts_backfill(vec_avg(positive_event_count_section), 100), ts_backfill(vec_avg(section_negative_event_count), 100)), 20))
    
    +rank(subtract(ts_backfill(vec_avg(nws3_scores_finupnormscr), 100), ts_backfill(vec_avg(nws3_scores_findownnormscr), 100)))
    
    """
    
)
#+rank(signed_power(ts_backfill(vec_avg(section_sentiment_score_3), 100), 3))
#+rank(ts_zscore(signed_power(multiply(divide(ts_backfill(mdl162_252_a_adbs, 100), ts_backfill(mdl162_66_q_adbs, 100)), divide(ts_backfill(mdl162_621_a_dtns, 100), ts_backfill(mdl162_66_q_dtns, 100))), 2), 20))


[simulate] launching: 

    rank(ts_zscore(divide(ts_backfill(vec_avg(positive_event_count_section), 100), ts_backfill(vec_avg(section_negative_event_count), 100)), 20))

    +rank(subtract(ts_backfill(vec_avg(nws3_scores_finupnormscr), 100), ts_backfill(vec_avg(nws3_scores_findownnormscr), 100)))

    
[simulate] USA/TOP3000 | delay 1 | decay 2 | neu INDUSTRY | max_trade OFF
[simulate] OK -> alpha_id QPbj7A3w
ALPHA REPORT  |  id: QPbj7A3w
link        : https://platform.worldquantbrain.com/alpha/QPbj7A3w
settings    : USA/TOP3000 | delay 1 | decay 2 | neutralization INDUSTRY
expression  : 

    rank(ts_zscore(divide(ts_backfill(vec_avg(positive_event_count_section), 100), ts_backfill(vec_avg(section_negative_event_count), 100)), 20))

    +rank(subtract(ts_backfill(vec_avg(nws3_scores_finupnormscr), 100), ts_backfill(vec_avg(nws3_scores_findownnormscr), 100)))

    

--- performance by year (10 rows + aggregate) ---


,sharpe,fitness,turnover,returns,drawdown,margin,year,stage
0,0.89,0.22,0.54,3.27%,4.67%,0.000122,ALL (IS),
1,-0.84,-0.19,0.49,-2.48%,4.67%,-0.000100,2014,IS
2,2.59,1.09,0.46,8.22%,1.84%,0.000354,2015,IS
3,1.41,0.45,0.47,4.69%,3.07%,0.000200,2016,IS
4,2.89,1.28,0.45,8.94%,1.42%,0.000395,2017,IS
5,1.84,0.59,0.54,5.63%,1.87%,0.000208,2018,IS
6,0.42,0.07,0.53,1.44%,2.07%,0.000054,2019,IS
7,0.42,0.08,0.60,2.02%,3.25%,0.000067,2020,IS
8,0.52,0.10,0.60,2.23%,4.19%,0.000075,2021,IS
9,0.53,0.10,0.64,2.37%,2.63%,0.000074,2022,IS



--- summary & submission gates ---


,metric,value,limit,status
0,sharpe (IS),0.89,NaN,NaN
1,fitness (IS),0.22,NaN,NaN
2,turnover (IS),53.83%,NaN,NaN
3,returns (IS),3.27%,NaN,NaN
4,drawdown (IS),4.67%,NaN,NaN
5,margin (IS),0.000122,NaN,NaN
6,self correlation,0.1225,0.7,PASS
7,prod correlation,0.5,0.7,PASS


top-5 self-correlated alphas:


,id,region,correlation,sharpe,fitness
0,MQ1JWoM,USA,0.1225,1.73,1.00
1,O9Qpkdd,USA,0.1026,1.51,1.46
2,6EmznvL,USA,0.0958,1.49,1.17
3,KLX5GKN,USA,0.0863,1.58,1.01
4,ZoY70M3,USA,0.0737,1.73,1.00


In [5]:
# ============================================================================
# FULL CATALOG CRAWLER — every dataset + every datafield for REGION/UNIVERSE/DELAY
# Output:  catalog/datasets_<tag>.csv        (all datasets, flattened)
#          catalog/datafields_<tag>.csv      (all datafields, flattened)
#          catalog/agent_catalog_<tag>.csv   (compact agent-ready subset of columns)
# Resumable: if interrupted (kernel death, network, rate limit), just re-run this
#            cell — it continues from the last checkpoint.
# ============================================================================
import os, time
import pandas as pd
import ace_lib as ace
import helpful_functions as hf

# ----------------------------- CONFIG ---------------------------------------
REGION, UNIVERSE, DELAY = "ASI", "TOP500", 1
INSTRUMENT   = "EQUITY"
PAGE         = 50     # /data-fields server page size (max allowed)
PAUSE        = 1    # seconds between pages (rate-limit safety)
CKPT_PAGES   = 20     # checkpoint every N pages
AUTO_RESUME  = True

OUT_DIR = "catalog"
os.makedirs(OUT_DIR, exist_ok=True)
tag      = f"{REGION}_{UNIVERSE}_D{DELAY}"
ds_csv   = f"{OUT_DIR}/datasets_{tag}.csv"
df_csv   = f"{OUT_DIR}/datafields_{tag}.csv"
ckpt_pkl = f"{OUT_DIR}/_partial_{tag}.pkl"
off_txt  = f"{OUT_DIR}/_offset_{tag}.txt"

def _flatten(rows):
    """Raw API dicts -> flat DataFrame (dataset/category/subcategory expanded)."""
    df = pd.DataFrame(rows)
    for col in ("dataset", "category", "subcategory"):
        if col in df.columns and len(df) and isinstance(df[col].iloc[0], dict):
            df = pd.concat([df.drop(columns=[col]),
                            df[col].apply(pd.Series).add_prefix(f"{col}_")], axis=1)
    return df

def _save_ckpt(rows, offset):
    pd.DataFrame(rows).to_pickle(ckpt_pkl)
    with open(off_txt, "w") as f:
        f.write(str(offset))

# --------------------- 1. ALL datasets (single known-good call) -------------
print("crawling datasets...")
datasets_df = ace.get_datasets(s, INSTRUMENT, region=REGION, delay=DELAY, universe=UNIVERSE)
datasets_df.to_csv(ds_csv, index=False)
print(f"  datasets: {len(datasets_df)} -> {ds_csv}")

# --------------------- 2. ALL datafields (paginated + resumable) ------------
base = (f"{ace.brain_api_url}/data-fields?instrumentType={INSTRUMENT}"
        f"&region={REGION}&delay={DELAY}&universe={UNIVERSE}")

rows, offset = [], 0
if AUTO_RESUME and os.path.exists(off_txt) and os.path.exists(ckpt_pkl):
    offset = int(open(off_txt).read().strip() or 0)
    rows = pd.read_pickle(ckpt_pkl).to_dict("records")
    print(f"resuming from offset {offset} ({len(rows)} fields already crawled)")

total, page = None, offset // PAGE
while True:
    url = f"{base}&limit={PAGE}&offset={offset}"
    r = s.get(url)
    while r.headers.get("Retry-After"):              # server says "come back later"
        time.sleep(float(r.headers["Retry-After"]))
        r = s.get(url)
    if r.status_code != 200:
        raise ValueError(f"API error {r.status_code}: {r.text[:200]}")
    try:
        ace._check_rate_limit(r)                     # sleeps if rate limit is near
    except AttributeError:
        pass

    j     = r.json()
    batch = j.get("results", [])
    total = j.get("count", total or len(rows) + len(batch))
    rows.extend(batch)
    page   += 1
    offset += len(batch)

    print(f"\r  page {page}: {len(rows)}/{total} fields ({len(rows)/total:.0%})", end="")

    if len(batch) < PAGE or len(rows) >= total:      # last page
        break
    if page % CKPT_PAGES == 0:                       # periodic checkpoint
        _save_ckpt(rows, offset)
    time.sleep(PAUSE)
print()

# --------------------- 3. flatten, dedupe, save ------------------------------
datafields_df = _flatten(rows).drop_duplicates(subset="id").reset_index(drop=True)
datafields_df.to_csv(df_csv, index=False)

agent_cols = [c for c in ["id", "dataset_id", "dataset_name", "category_id",
                          "subcategory_id", "type", "coverage", "userCount",
                          "alphaCount", "description"] if c in datafields_df.columns]
datafields_df[agent_cols].to_csv(f"{OUT_DIR}/agent_catalog_{tag}.csv", index=False)

for f in (ckpt_pkl, off_txt):                        # crawl complete -> clean up
    if os.path.exists(f):
        os.remove(f)

print(f"\nDONE")
print(f"  datasets   : {len(datasets_df):>6} -> {ds_csv}")
print(f"  datafields : {len(datafields_df):>6} -> {df_csv}")
print(f"  agent cols : {len(agent_cols)} columns -> {OUT_DIR}/agent_catalog_{tag}.csv")

print("\nfields per dataset (top 20):")
print(datafields_df.groupby("dataset_id").size().sort_values(ascending=False).head(20))
display(datafields_df[agent_cols].head(10))

crawling datasets...
  datasets: 160 -> catalog/datasets_ASI_TOP500_D1.csv
  page 200: 10000/10000 fields (100%)

DONE
  datasets   :    160 -> catalog/datasets_ASI_TOP500_D1.csv
  datafields :  10000 -> catalog/datafields_ASI_TOP500_D1.csv
  agent cols : 10 columns -> catalog/agent_catalog_ASI_TOP500_D1.csv

fields per dataset (top 20):
dataset_id
analyst15                    1413
analyst69                     978
fundamental72                 751
analyst10                     643
analyst_revision_horizons     548
fundamental28                 522
analyst_consensus             346
analyst44                     344
fundamental17                 344
fundamental23                 316
fundamental6                  295
ai_equity_alpha               276
continuation_score            272
predictive_starmine           268
analyst4                      266
global_seasonal_model         192
model26                       181
pattern_scores                156
intraday_pv_feats             156
ana

,id,dataset_id,dataset_name,category_id,subcategory_id,type,coverage,userCount,alphaCount,description
0,absolute_price_change_today_2,analyst45,Analyst Trade Ideas,analyst,analyst-analyst-estimates,VECTOR,0.4293,0,0,Change in the stock price since the previous d...
1,accounting_forensics_factor,ai_equity_alpha,AI Equity Alpha Forecasts,model,model-mlai-models,VECTOR,1.0000,1,1,Analysis of accounts for signs of financial tr...
2,accounting_forensics_factor_2,ai_equity_alpha,AI Equity Alpha Forecasts,model,model-mlai-models,VECTOR,1.0000,1,1,Alternative analysis of accounts for signs of ...
3,accounting_forensics_factor_3,ai_equity_alpha,AI Equity Alpha Forecasts,model,model-mlai-models,VECTOR,1.0000,0,0,Analysis of accounts for signs of financial tr...
4,accounting_forensics_factor_4,ai_equity_alpha,AI Equity Alpha Forecasts,model,model-mlai-models,VECTOR,0.9699,0,0,Analysis of accounts for signs of trouble and ...
5,accounting_forensics_factor_41,ai_equity_alpha,AI Equity Alpha Forecasts,model,model-mlai-models,VECTOR,0.9699,0,0,Subtle analysis of accounts for signs of troub...
6,accounting_forensics_factor_5,ai_equity_alpha,AI Equity Alpha Forecasts,model,model-mlai-models,VECTOR,0.9699,0,0,Subtle analysis of accounts for signs of troub...
7,accounting_forensics_factor_51,ai_equity_alpha,AI Equity Alpha Forecasts,model,model-mlai-models,VECTOR,0.9695,0,0,Analysis of accounts for signs of trouble and ...
8,accounts_notes_payable_ard,fundamental72,Comprehensive Fundamental Data,fundamental,fundamental-fundamental-data,VECTOR,0.2589,0,0,Accounts and notes payable as reported in the ...
9,accounts_notes_receivable_ard,fundamental72,Comprehensive Fundamental Data,fundamental,fundamental-fundamental-data,VECTOR,0.2570,0,0,Accounts and notes receivable as reported.
